# 01 · Problem Framing and Data Acquisition
**Milestone 1 — Kenyan Retail Customer & Sales Intelligence Capstone**

This notebook frames the business problem, sets objectives and measurable success indicators, lists the data needed,
and gives a first look at the acquired data (the synthetic retail tables plus verified Kenyan public-holiday data).

| Notebook | Purpose |
|---|---|
| **01 Problem framing** (this one) | Why the project exists and what data it needs |
| 02 Data generation | How the synthetic data was built and checked |
| 03 Data cleaning | Milestone 2 |
| 04 EDA · 05 Statistics · 06 Modelling · 07 Evaluation | Milestones 3, 5 and 6 |

> **Data notice:** the retailer *Duka Bora* and all of its transactions, customers, brands and prices are synthetic.
> Only the public-holiday file is real public data.

## 1. Business problem

Duka Bora is a growing six-store supermarket chain in Nairobi, Kiambu, Mombasa, Kisumu and Nakuru.
It records every sale, but management decisions on stock, promotions and customer retention are still based on intuition.
Management has four questions:

1. **Who buys what, where and when?** Which customers, products, stores and time periods make up sales.
2. **Which products and stores drive revenue *and profit*?** High revenue does not always mean high margin.
3. **Which customers are drifting away?** Loyal customers who have stopped buying are revenue that is quietly being lost.
4. **What demand should we expect next month?** This is needed to plan stock and staffing.

**Problem statement:** *Duka Bora lacks an evidence base for commercial decisions. This project turns two years of
transaction data (January 2024 to December 2025) into descriptive insight, statistically tested findings, a
next-month demand forecast and a list of at-risk customers, all delivered through a management dashboard.*

## 2. Project objectives

| # | Objective | Delivered in |
|---|---|---|
| O1 | Build a reproducible, documented, relational retail dataset (3,000 sales lines) joined to public holiday data | M1 |
| O2 | Clean and validate the data, with a before-and-after quality report | M2 |
| O3 | Describe sales, product, customer, store and promotion performance | M3 |
| O4 | Answer core business questions with SQL in a normalized PostgreSQL database (including RFM and cohorts) | M4 |
| O5 | Test whether promotions, stores and categories show statistically significant differences | M5 |
| O6 | Forecast next month's revenue against a baseline, and identify at-risk customers | M6 |
| O7 | Communicate the findings in a 5-page Power BI dashboard | M7 |
| O8 | Document everything so another analyst can reproduce it | M8 |

## 3. Analytical questions

| ID | Question | Method | Milestone |
|---|---|---|---|
| Q1 | Which products, categories and stores generate the most revenue and profit? | Aggregation, ranking | M3, M4 |
| Q2 | How do sales vary by month, weekday, hour, payday window and public holidays? | Time-series EDA, date SQL | M3, M4 |
| Q3 | What is the average order value, and how does it differ by store and payment method? | Order-level aggregation | M3, M4 |
| Q4 | Who are the top 10 and the most valuable customers? | Ranking, RFM | M4 |
| Q5 | How well are customers retained month to month after their first purchase? | Cohort analysis | M4 |
| Q6 | Do promotional discounts significantly increase sales? | Mann-Whitney U / Welch t-test, bootstrap CI | M5 |
| Q7 | Does average customer spending differ across stores? | ANOVA / Kruskal-Wallis + post-hoc | M5 |
| Q8 | Does purchasing behaviour differ across product categories? | Chi-square, Kruskal-Wallis | M5 |
| Q9 | What revenue should management expect next month? | Baselines vs exponential smoothing vs Random Forest | M6 |
| Q10 | Which customers are at risk of churning, using a defensible inactivity threshold? | Recency threshold + classifier | M6 |

## 4. Success indicators (how we will know the project worked)

| Area | Indicator | Target |
|---|---|---|
| Data | Rows in the raw sales file | **Exactly 3,000** |
| Data | Foreign keys that resolve after cleaning | **100%** |
| Cleaning | Injected issues detected (audited against the log at the end of M2) | **100%** |
| Cleaning | Unexplained duplicates left in the cleaned data | **0** |
| SQL | Scripts that run without error on PostgreSQL 18 | **7 of 7** |
| Statistics | Tests reported with assumption checks, p-value and confidence interval | **3 of 3** |
| Forecasting | Test-set MAE of the chosen model vs the naive baseline | **Lower than the baseline** (reported honestly if not) |
| Churn | Inactivity threshold justified from purchase-gap data | Documented and defended |
| Communication | Dashboard pages with the required visuals and slicers | **5 of 5** |
| Reproducibility | Notebooks run top to bottom on a clean environment | **All** |

## 5. Data required

| Dataset | Why it is needed | Source |
|---|---|---|
| `sales_transactions` | Fact table: what was sold, where, when, to whom, at what price and profit | Synthetic (notebook 02) |
| `customers` | Segmenting customers, cohorts, RFM | Synthetic |
| `products` | Category and product performance, unit costs | Synthetic |
| `stores` | Store and county comparisons | Synthetic |
| `promotions` | Measuring the effect of discounts | Synthetic |
| `calendar` | Weekday, payday and holiday features for EDA and forecasting | Generated from dates and holiday data |
| `kenya_public_holidays_2024_2025` | Holiday effects on demand | **Real public data**: Public Holidays Act and Kenya Gazette |

Full column definitions are in [`docs/data_dictionary.md`](../docs/data_dictionary.md).
Provenance and licences are in [`docs/data_sources.md`](../docs/data_sources.md).

## 6. First look at the acquired data

In [1]:
# --- Setup: make the project's "src" package importable from this notebook ---
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.visualization import style

style.apply_style()
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
# Load every raw table. All columns are read as-is: no cleaning happens in this notebook.
table_names = ["sales_transactions", "customers", "products", "stores", "promotions", "calendar"]
raw = {name: pd.read_csv(config.RAW_DIR / f"{name}.csv") for name in table_names}
holidays = pd.read_csv(config.EXTERNAL_DIR / "kenya_public_holidays_2024_2025.csv")

inventory = pd.DataFrame({
    "table": table_names + ["kenya_public_holidays (external)"],
    "rows": [len(raw[n]) for n in table_names] + [len(holidays)],
    "columns": [raw[n].shape[1] for n in table_names] + [holidays.shape[1]],
})
inventory

,table,rows,columns
0,sales_transactions,3000,16
1,customers,1800,7
2,products,40,7
3,stores,6,7
4,promotions,20,7
5,calendar,731,14
6,kenya_public_holidays (external),28,5


The main sales table has **exactly 3,000 rows**, which meets the assignment requirement. The dimension tables hold
1,800 customers, 40 products, 6 stores, 20 promotions and 731 calendar days (every day of 2024 and 2025).

In [3]:
# Preview the fact table
raw["sales_transactions"].head()

,transaction_id,order_id,transaction_date,transaction_time,customer_id,store_id,product_id,quantity,unit_price,unit_cost,discount_percentage,discount_amount,total_amount,profit,payment_method,promotion_id
0,TXN00001,ORD00001,2024-01-01,10:34,C0723,S06,P001,5,185.00,162.00,0.00,0.00,925.0,115.00,Cash,NaN
1,TXN00002,ORD00002,2024-01-01,11:29,C0724,S05,P002,1,195.00,170.00,0.00,0.00,195.0,25.00,Cash,NaN
2,TXN00003,ORD00003,2024-01-01,11:58,C0722,S01,P029,1,380.00,260.00,0.00,0.00,380.0,120.00,M-Pesa,NaN
3,TXN00004,ORD00004,2024-01-01,15:19,C0593,S03,P022,1,180.00,130.00,0.00,0.00,180.0,50.00,Cash,NaN
4,TXN00005,ORD00005,2024-01-02,14:23,C0661,S04,P007,1,70.00,60.00,10.00,7.00,63.0,3.00,M-Pesa,PR01


In [4]:
# Preview the four main dimension tables
for name in ["customers", "products", "stores", "promotions"]:
    print(f"\n=== {name} ===")
    display(raw[name].head(3))


=== customers ===


,customer_id,gender,age_band,home_county,registration_date,loyalty_member,preferred_store_id
0,C0001,Female,25-34,Nairobi,2020-01-04,Yes,S03
1,C0002,Female,35-44,Nakuru,2020-01-05,No,S06
2,C0003,Male,45-54,Nairobi,2020-01-06,Yes,S01



=== products ===


,product_id,product_name,category,brand,pack_size,selling_price_kes,unit_cost_kes
0,P001,Maize Flour,Food and Groceries,Shamba Gold,2 kg,185,162
1,P002,Wheat Flour,Food and Groceries,Shamba Gold,2 kg,195,170
2,P003,Long Grain Rice,Food and Groceries,Pwani Grains,5 kg,1050,920



=== stores ===


,store_id,store_name,town,county,store_format,opening_date,floor_area_sqm
0,S01,Duka Bora Nairobi CBD,Nairobi CBD,Nairobi,Flagship,2016-03-01,2400
1,S02,Duka Bora Westlands,Westlands,Nairobi,Flagship,2018-08-15,2100
2,S03,Duka Bora Thika,Thika,Kiambu,Compact,2021-02-01,900



=== promotions ===


,promotion_id,promotion_name,promotion_type,start_date,end_date,target_category,discount_percentage
0,PR01,New Year Grocery Saver,Seasonal,2024-01-02,2024-01-14,Food and Groceries,10.00
1,PR02,Back-to-School January,Back-to-school,2024-01-02,2024-01-21,Clothing,15.00
2,PR03,Valentine's Self-Care Week,Seasonal,2024-02-07,2024-02-14,Personal Care,12.00


### 6.1 Do the tables join correctly?

Each foreign key in the sales table should point to an existing row in its dimension table. This is checked on the
raw data. Blank `customer_id` and `promotion_id` values are counted separately, because a blank is not a broken link.

In [5]:
sales = raw["sales_transactions"]
fk_checks = [
    ("store_id", "stores", "store_id"),
    ("product_id", "products", "product_id"),
    ("customer_id", "customers", "customer_id"),
    ("promotion_id", "promotions", "promotion_id"),
]
rows = []
for fk, table, pk in fk_checks:
    values = sales[fk].dropna()
    rows.append({
        "foreign_key": fk,
        "references": f"{table}.{pk}",
        "non_blank_values": len(values),
        "blank_values": sales[fk].isna().sum(),
        "unmatched_values": (~values.isin(raw[table][pk])).sum(),
    })
pd.DataFrame(rows)

,foreign_key,references,non_blank_values,blank_values,unmatched_values
0,store_id,stores.store_id,3000,0,0
1,product_id,products.product_id,3000,0,0
2,customer_id,customers.customer_id,2975,25,0
3,promotion_id,promotions.promotion_id,316,2684,0


**Interpretation:** every non-blank key matches a row in its dimension table, so the schema is relationally sound.
- About 2,700 lines have no `promotion_id`. This is expected, because most sales are not on promotion.
- The **25 blank customer IDs** are a data-quality issue to handle in Milestone 2. These are likely walk-in sales where no loyalty card was scanned.

### 6.2 Public context data: Kenyan public holidays 2024–2025

Holiday dates come from the Public Holidays Act and Kenya Gazette special notices (sources are cited per row).
The checks below confirm two rules in code rather than by eye:
- the weekday of each holiday is shown;
- under the Public Holidays Act, every holiday that falls on a **Sunday** must have an "(observed)" holiday on the Monday.

In [6]:
holidays["date"] = pd.to_datetime(holidays["date"])
holidays["weekday"] = holidays["date"].dt.day_name()
display(holidays[["date", "weekday", "holiday_name", "holiday_type"]])

sunday_holidays = holidays[holidays["weekday"] == "Sunday"]
for _, h in sunday_holidays.iterrows():
    next_day = h["date"] + pd.Timedelta(days=1)
    has_substitute = next_day in set(holidays["date"])
    print(f"{h['date'].date()} {h['holiday_name']} falls on a Sunday -> Monday substitute present: {has_substitute}")
print("\nHoliday counts by type:\n", holidays["holiday_type"].value_counts().to_string())

,date,weekday,holiday_name,holiday_type
0,2024-01-01,Monday,New Year's Day,Statutory
1,2024-03-29,Friday,Good Friday,Statutory
2,2024-04-01,Monday,Easter Monday,Statutory
3,2024-04-10,Wednesday,Idd-ul-Fitr,Special gazette notice
4,2024-05-01,Wednesday,Labour Day,Statutory
5,2024-05-10,Friday,National Tree Growing Day,Special gazette notice
6,2024-06-01,Saturday,Madaraka Day,Statutory
7,2024-06-17,Monday,Eid-al-Adha,Special gazette notice
8,2024-10-10,Thursday,Mazingira Day,Statutory
9,2024-10-20,Sunday,Mashujaa Day,Statutory


2024-10-20 Mashujaa Day falls on a Sunday -> Monday substitute present: True
2025-06-01 Madaraka Day falls on a Sunday -> Monday substitute present: True

Holiday counts by type:
 holiday_type
Statutory                      20
Special gazette notice          6
Statutory Sunday substitute     2


**Interpretation:** there are 28 holiday dates across the two years: 20 statutory, 2 Sunday substitutes and 6 special
gazetted holidays. Both Sunday holidays have their Monday substitute, so the file follows the Act.
The holidays are joined to the sales data through `calendar.date`. In Milestone 3 we will check whether shopping rises
on the eve of a holiday.

**Business implication:** holidays such as Idd-ul-Fitr and the 2025 state funeral holiday were announced only a few
days in advance. Demand plans should allow for short-notice holidays, not only the fixed statutory ones.

## 7. Scope, assumptions and ethics

- **Synthetic data:** this makes it possible to share the full project publicly without exposing customer data.
  The trade-off is that findings describe the simulated business, not a real chain.
- **Scale:** the 3,000 lines should be read as a *sample of loyalty-tracked transactions*, not the chain's full turnover.
  Absolute revenue figures are therefore small. Comparisons and patterns are what matter.
- **No personal data:** customers have no names, phone numbers or ID numbers.
- **KNBS CPI and census population data** were removed from scope at the project owner's request.
  No official statistic is simulated.
- **No causal claims:** the data is observational, so later milestones report associations unless a test design supports more.

## 8. Next step

Notebook **02** explains exactly how the data was generated and checks that the intended business patterns are present.